# MIA-07 · Inferencia de fisuras en Google Colab

Esta demostración carga los checkpoints BEST de A0 y A1 de la semilla de referencia 20261026. Con la misma fotografía de Crack-Seg, se muestran las máscaras que generan los dos modelos. El umbral de segmentación es 0,5 y la resolución de trabajo es 416 × 416.

En Colab elige **Entorno de ejecución → Ejecutar todas**. La fotografía de ejemplo y los pesos se obtienen del repositorio. En la segunda celda se puede activar la carga de una fotografía propia.

La fotografía de ejemplo corresponde a `2323.rf.06768caef93b3a44d3c59325a03230eb` de [Crack-Seg](https://docs.ultralytics.com/datasets/segment/crack-seg/) y su fuente [Roboflow Universe](https://universe.roboflow.com/university-bswxt/crack-bphdr) declara CC0 1.0. La salida delimita regiones de fisura visible en la imagen.


In [ ]:
from pathlib import Path
import subprocess
import sys

repo_url = 'https://github.com/nick5uni/mia07-segmentacion-fisuras.git'
root = Path.cwd()
if not (root / 'src/inferencia.py').is_file():
    root = root / 'mia07-segmentacion-fisuras'
    if not (root / 'src/inferencia.py').is_file():
        subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(root)], check=True)

sys.path.insert(0, str(root / 'src'))
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from inferencia import predict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    torch.set_num_threads(4)
print(f'PyTorch {torch.__version__} | Dispositivo {device}')


In [ ]:
# Cambiar a True para escoger una fotografía propia en Colab.
usar_foto_propia = False
foto = root / 'data/ejemplo/fotografia.jpg'
if usar_foto_propia:
    from google.colab import files
    archivos = files.upload()
    if len(archivos) != 1:
        raise ValueError('Selecciona una sola fotografía.')
    foto = Path(next(iter(archivos)))

predicciones = {m: predict(foto, model_name=m, device=device) for m in ('A0', 'A1')}
print('Inferencia terminada con los dos checkpoints BEST.')


In [ ]:
figura, ejes = plt.subplots(2, 3, figsize=(11, 7))
for fila, (nombre, (rgb, probabilidades, mascara)) in enumerate(predicciones.items()):
    original = np.asarray(rgb)
    superpuesta = original.copy()
    superpuesta[mascara] = 0.4 * original[mascara] + 0.6 * np.array([255, 40, 40])
    ejes[fila, 0].imshow(original)
    ejes[fila, 0].set_title('Fotografía · ' + nombre)
    ejes[fila, 1].imshow(mascara, cmap='gray', vmin=0, vmax=1)
    ejes[fila, 1].set_title('Máscara predicha · ' + nombre)
    ejes[fila, 2].imshow(superpuesta)
    ejes[fila, 2].set_title('Predicción sobre la foto · ' + nombre)
    for eje in ejes[fila]:
        eje.axis('off')
figura.tight_layout()
plt.show()
